# 🫀 心臟衰竭生存風險預測 Gradio Web UI 實作教學

> 對象：修習機器學習模型部署與 Web UI 開發的學員
> 對應程式碼：`app.py` 第 112~191 行 (`Gradio UI` 區塊)
>
> 本 Notebook 帶你拆解 Gradio 互動式急診重症監控儀表板：從輸入滑桿與勾選鈕、急診風險警示燈、到使用 Matplotlib 動態繪製 11 項臨床特徵重要性 (Feature Importances) 水平長條圖。

## 🎯 學習目標

- [ ] 掌握 Gradio `Blocks` 雙欄醫療排版架構（左側 11 項指標輸入、右側診斷警戒卡與特徵圖表）
- [ ] 能將 11 項臨床特徵正確轉換為 NumPy 陣列並傳遞給隨機森林模型
- [ ] 理解如何在 Gradio 中使用 Matplotlib 動態繪製水平長條圖呈現 Feature Importances
- [ ] 掌握三級急診警示訊息與標籤連動輸出
- [ ] 掌握如何將 Gradio App 掛載至 FastAPI (`gr.mount_gradio_app`)

In [ ]:
# ============================================
# 0. 載入套件與模型
# ============================================
import os, joblib
import numpy as np
import matplotlib.pyplot as plt
import gradio as gr

MODEL_PATH = "heart_failure_model.joblib"
if not os.path.exists(MODEL_PATH):
    from train_save import train_and_save_model
    train_and_save_model(model_path=MODEL_PATH)

meta = joblib.load(MODEL_PATH)
model = meta["model"]
feature_importances = meta["feature_importances"]
print("✅ 心臟衰竭模型載入完成！")

---
## Part 1：定義 Gradio 推論與特徵重要性繪圖核心函數 (`gradio_predict`)

In [ ]:
def gradio_predict(
    age, anaemia, cpk, diabetes, ef, hbp, platelets,
    serum_creatinine, serum_sodium, sex, smoking
):
    anaemia_val = 1 if anaemia else 0
    diabetes_val = 1 if diabetes else 0
    hbp_val = 1 if hbp else 0
    sex_val = 1 if sex == "男性" else 0
    smoking_val = 1 if smoking else 0
    
    raw_features = np.array([[
        age, anaemia_val, cpk, diabetes_val, ef, hbp_val,
        platelets, serum_creatinine, serum_sodium, sex_val, smoking_val
    ]])
    
    prob = float(model.predict_proba(raw_features)[0, 1])
    prob_percent = f"{prob * 100:.1f}%"
    
    if prob >= 0.6:
        badge = "🔴 【重症高死亡風險】"
        alert = "急診警示：心腎功能嚴重受損，建議即刻送加護病房 (ICU) 或安排心導管/強心藥物處置。"
    elif prob >= 0.35:
        badge = "🟡 【中度監控風險】"
        alert = "急診警示：心臟收縮功能受損，需密切追蹤電解質平衡與每日體重尿量變化。"
    else:
        badge = "🟢 【相對平穩低風險】"
        alert = "急診警示：各項器官灌流指數尚可，持續維持常規慢性心衰竭處置。"
        
    # 繪製特徵重要性水平長條圖
    fig, ax = plt.subplots(figsize=(6, 4))
    sorted_items = sorted(feature_importances.items(), key=lambda x: x[1])
    feat_names = [x[0] for x in sorted_items]
    importances = [x[1] for x in sorted_items]
    bars = ax.barh(feat_names, importances, color="crimson")
    ax.set_title("隨機森林臨床指標重要性 (Feature Importances)", fontsize=11)
    ax.set_xlabel("權重貢獻度", fontsize=10)
    for bar in bars:
        w = bar.get_width()
        ax.text(w + 0.005, bar.get_y() + bar.get_height()/2, f"{w:.3f}", va="center", fontsize=8)
    plt.tight_layout()
    
    return prob_percent, badge, alert, fig

# 測試執行一次
p, b, a, f = gradio_predict(65, False, 160, False, 35, True, 250000, 1.8, 136, "男性", False)
print(f"機率: {p}, 標籤: {b}")

---
## Part 2：建構 Gradio Blocks Web 介面

In [ ]:
with gr.Blocks(title="心臟衰竭生存風險預測系統", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🫀 智慧急診：心臟衰竭病患重症風險即時預測系統")
    gr.Markdown("利用 **隨機森林集成演算法 (Random Forest)**，即時預估病患院內死亡風險並解析核心急診生化指標權重。")
    
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 🩺 病患臨床生理檢驗輸入")
            in_age = gr.Slider(18, 100, 65, step=1, label="年齡 (歲)")
            in_ef = gr.Slider(10, 80, 35, step=1, label="射血分數 EF (%)")
            in_creatinine = gr.Slider(0.5, 9.0, 1.8, step=0.1, label="血清肌酸酐 (mg/dL)")
            in_cpk = gr.Slider(20, 5000, 250, step=10, label="肌酸激酶 CPK (mcg/L)")
            in_sodium = gr.Slider(110, 150, 136, step=1, label="血清鈉 (mEq/L)")
            in_platelets = gr.Slider(30000, 800000, 250000, step=5000, label="血小板計數 (cells/mL)")
            in_sex = gr.Radio(["男性", "女性"], value="男性", label="生理性別")
            with gr.Row():
                in_anaemia = gr.Checkbox(label="貧血", value=False)
                in_diabetes = gr.Checkbox(label="糖尿病", value=False)
                in_hbp = gr.Checkbox(label="高血壓", value=True)
                in_smoking = gr.Checkbox(label="抽菸", value=False)
            btn_predict = gr.Button("評估生存風險", variant="primary")
            
        with gr.Column(scale=1):
            gr.Markdown("### 📊 急診診斷與特徵權重")
            out_prob = gr.Label(label="惡化死亡預估機率")
            out_badge = gr.Textbox(label="風險等級", interactive=False)
            out_alert = gr.Textbox(label="急診處置指引", interactive=False, lines=3)
            out_plot = gr.Plot(label="生化指標權重排名")
            
    btn_predict.click(
        fn=gradio_predict,
        inputs=[
            in_age, in_anaemia, in_cpk, in_diabetes, in_ef, in_hbp,
            in_platelets, in_creatinine, in_sodium, in_sex, in_smoking
        ],
        outputs=[out_prob, out_badge, out_alert, out_plot]
    )

print("✅ Gradio Blocks 介面定義完成！可執行 demo.launch() 啟動測試。")